# Note
- This notebook contains code to train baseline models and the first proposed model of CNN + Transformer.
- Each run prints the model architecture and parameter count before training, and the final test-set metrics (Macro F1, AUROC, Precision, Recall, and per-class AUROC) at the end.

# Import
- Load helpers from `utils.py`, the baselines from `baseline.py`, and the proposed model from `cnn_transformer.py`.

In [1]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from utils import DEVICE, SUPERCLASSES, preprocess_signals, extract_features, load_preprocessed, \
    PTBXLDataset, train_one_epoch, evaluate, tune_thresholds, make_warmup_cosine, model_summary
from baseline import NUM_CLASSES, make_loaders, run_xgboost, train_torch_baseline,CNN, ResNet1D, Transformer
from cnn_transformer import CnnTransformer, train_proposed

# Data Preprocessing
- `RUN_FEATURE_EXTRACTION = True` will read the raw PTB-XL records, applies the train/val/test split, and caches the preprocessed signals and the 179 hand-crafted features to `./preprocessed/`. Set it to `False` will run to load the cached arrays.
- This step only needs to run once and takes about 20 minutes.

In [2]:
# preprocess signal and hand-crafted features
RUN_FEATURE_EXTRACTION = True  
preprocess_signals()
if RUN_FEATURE_EXTRACTION:
    extract_features()

PTB-XL Signal Preprocessing
Preprocessed files already exist in ./preprocessed/
Delete them to regenerate. Skipping.
Hand-crafted Feature Extraction (XGBoost, unnormalized signals)
Feature files already exist. Delete to regenerate. Skipping.


# Baselines
- Train and evaluate baseline models: XGBoost, 1D CNN, ResNet1D, and the Transformer.
- Each model saves its best checkpoint, tuned per-class thresholds, and test metrics to its own `./checkpoint_*/` directory.

In [3]:
def run_baseline(model, title, checkpoint_dir, *, augment_train, epochs, lr,
                 patience, warmup_epochs=0, weight_decay=1e-4, grad_clip=1.0,
                 banner=None):
    print("\n" + "=" * 60)
    print(banner or f"BASELINE — {title}")
    print("=" * 60)
    cfg = {"checkpoint_dir": checkpoint_dir, "epochs": epochs, "lr": lr,
           "weight_decay": weight_decay, "patience": patience,
           "grad_clip": grad_clip, "warmup_epochs": warmup_epochs}
    loaders = make_loaders(augment_train=augment_train)
    return train_torch_baseline(model, cfg, loaders, title)

BASELINES = {
    "cnn": dict(
        model=lambda: CNN(num_classes=NUM_CLASSES),
        title="1D CNN", banner="Baseline 2 - 1D CNN",
        checkpoint_dir="./checkpoint_cnn", augment_train=False,
        epochs=30, lr=1e-3, weight_decay=1e-4, patience=7, warmup_epochs=0),
    "resnet1d": dict(
        model=lambda: ResNet1D(num_classes=NUM_CLASSES),
        title="ResNet1D", banner="Baseline 3 — ResNet1D",
        checkpoint_dir="./checkpoint_resnet", augment_train=True,
        epochs=40, lr=1e-3, weight_decay=1e-4, patience=8, warmup_epochs=0),
    "transformer": dict(
        model=lambda: Transformer(num_classes=NUM_CLASSES),
        title="Transformer", banner="Baseline 4 — Transformer",
        checkpoint_dir="./checkpoint_transformer", augment_train=True,
        epochs=40, lr=3e-4, weight_decay=1e-4, patience=10, warmup_epochs=4),
}


In [4]:
# Run all baselines and summarize
print(f"Device: {DEVICE}\n")

results = {"XGBoost": run_xgboost()}        
for name, spec in BASELINES.items():
    spec = dict(spec)
    spec["model"] = spec["model"]()         
    results[name] = run_baseline(**spec)

print("\n" + "=" * 60)
print("Baseline Summary (test set)")
print("=" * 60)
print(f"  {'Model':20s}  {'Macro F1':>9s}  {'Macro AUROC':>12s}")
for name, m in results.items():
    print(f"  {name:20s}  {m['f1']:>9.4f}  {m['auroc']:>12.4f}")

Device: cuda

Baseline 1 — XGBoost

Loading hand-crafted features...
  Train (17084, 179)  Val (2146, 179)  Test (2158, 179)
  Number of features: 179

Training one XGBoost model per class (One-vs-Rest)...
  [1/5] NORM   best_iter=236
  [2/5] MI     best_iter=439
  [3/5] STTC   best_iter=295
  [4/5] CD     best_iter=490
  [5/5] HYP    best_iter=399

Tuning per-class thresholds on validation set...
    NORM : 0.520
    MI   : 0.440
    STTC : 0.470
    CD   : 0.430
    HYP  : 0.460

Final Test Results — XGBoost
  Macro F1  : 0.7070
  Macro AUROC: 0.9015
  Macro Prec: 0.6783
  Macro Rec : 0.7403
  Per-class AUROC:
    NORM : 0.9238
    MI   : 0.8720
    STTC : 0.9158
    CD   : 0.8982
    HYP  : 0.8976

Artifacts saved to ../checkpoint_xgb/

Baseline 2 - 1D CNN
  Train: (17084, 12, 1000)  Val: (2146, 12, 1000)  Test: (2158, 12, 1000)
Model Architecture: 1D CNN
CNN(
  (features): Sequential(
    (0): Sequential(
      (0): Conv1d(12, 32, kernel_size=(7,), stride=(1,), padding=(3,))
      


Epoch 01/30  (lr=1.00e-03)
  Train  Loss=0.3498  F1=0.6165
  Val    Loss=0.3119  F1=0.6898  Prec=0.7530  Recall=0.6551  AUROC=0.9079
   checkpoint: best model saved (val F1: 0.6898)



Epoch 02/30  (lr=9.97e-04)
  Train  Loss=0.2997  F1=0.6973
  Val    Loss=0.3042  F1=0.7203  Prec=0.7481  Recall=0.7109  AUROC=0.9166
   checkpoint: best model saved (val F1: 0.7203)



Epoch 03/30  (lr=9.89e-04)
  Train  Loss=0.2845  F1=0.7172
  Val    Loss=0.2913  F1=0.7140  Prec=0.7766  Recall=0.6780  AUROC=0.9181
  Patience 1/7



Epoch 04/30  (lr=9.76e-04)
  Train  Loss=0.2741  F1=0.7276
  Val    Loss=0.2799  F1=0.7109  Prec=0.8013  Recall=0.6570  AUROC=0.9257
  Patience 2/7


...


Epoch 22/40  (lr=1.63e-04)
  Train  Loss=0.2498  F1=0.7574
  Val    Loss=0.3230  F1=0.6783  Prec=0.7906  Recall=0.6278  AUROC=0.9089
  Patience 7/10



Epoch 23/40  (lr=1.50e-04)
  Train  Loss=0.2467  F1=0.7626
  Val    Loss=0.3251  F1=0.6900  Prec=0.7901  Recall=0.6348  AUROC=0.9120
  Patience 8/10



Epoch 24/40  (lr=1.37e-04)
  Train  Loss=0.2459  F1=0.7639
  Val    Loss=0.3005  F1=0.6885  Prec=0.7908  Recall=0.6350  AUROC=0.9169
  Patience 9/10



Epoch 25/40  (lr=1.24e-04)
  Train  Loss=0.2423  F1=0.7647
  Val    Loss=0.3089  F1=0.6942  Prec=0.7942  Recall=0.6462  AUROC=0.9141
  Patience 10/10

Early stopping at epoch 25

Evaluating best model on test set...



FINAL TEST RESULTS — Transformer
  Loss      : 0.3137
  Macro F1  : 0.6879
  Macro AUROC: 0.9049
  Macro Prec: 0.7762
  Macro Rec : 0.6374
  Per-class AUROC:
    NORM : 0.9370
    MI   : 0.9008
    STTC : 0.9282
    CD   : 0.8819
    HYP  : 0.8765

Artifacts saved to ./checkpoint_transformer/

Baseline Summary (test set)
  Model                  Macro F1   Macro AUROC
  XGBoost                  0.7070        0.9015
  cnn                      0.7380        0.9266
  resnet1d                 0.7392        0.9225
  transformer              0.6879        0.9049


# Proposed Model 1: CNN + Transformer
- Train and evaluate CNN + Transformer model via `train_proposed(...)`.
- Artifacts are saved to `./checkpoint_proposed/`.

In [3]:
cfg = {
    "checkpoint_dir": "./checkpoint_proposed",
    "epochs": 50,
    "lr": 5e-4,
    "weight_decay": 1e-4,
    "patience": 12,
    "grad_clip": 1.0,
    "warmup_epochs": 4,
    "select_metric": "f1",  
}

loaders = make_loaders(augment_train=True)
model = CnnTransformer(num_classes=NUM_CLASSES)
test_metrics = train_proposed(model, cfg, loaders,
                              title="CNN + Transformer (Proposed model 1)")

  Train: (17084, 12, 1000)  Val: (2146, 12, 1000)  Test: (2158, 12, 1000)


/opt/conda/lib/python3.11/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


Model Architecture: CNN + Transformer (Proposed model 1)
CnnTransformer(
  (per_lead_cnn): PerLeadCNN(
    (cnn): Sequential(
      (0): Conv1d(12, 96, kernel_size=(15,), stride=(2,), padding=(7,), groups=12, bias=False)
      (1): BatchNorm1d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): Conv1d(96, 192, kernel_size=(7,), stride=(2,), padding=(3,), groups=12, bias=False)
      (5): BatchNorm1d(192, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU(inplace=True)
      (7): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (8): Conv1d(192, 384, kernel_size=(3,), stride=(1,), padding=(1,), groups=12, bias=False)
      (9): BatchNorm1d(384, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (10): ReLU(inplace=True)
    )
  )
  (lead_attention): SpatialLeadAttention


Epoch 01/50  (lr=1.25e-04)
  Train  Loss=0.3752  F1=0.5852
  Val    Loss=0.3429  F1=0.6729  Prec=0.7503  Recall=0.6306  AUROC=0.8969
  Val per-class AUROC: NORM=0.936  MI=0.874  STTC=0.917  CD=0.877  HYP=0.881
   checkpoint: best model saved (val f1: 0.6729)



Epoch 02/50  (lr=2.50e-04)
  Train  Loss=0.3210  F1=0.6747
  Val    Loss=0.3236  F1=0.7136  Prec=0.7546  Recall=0.6861  AUROC=0.9118
  Val per-class AUROC: NORM=0.938  MI=0.893  STTC=0.925  CD=0.908  HYP=0.895
   checkpoint: best model saved (val f1: 0.7136)


...


Epoch 27/50  (lr=2.67e-04)
  Train  Loss=0.1873  F1=0.8222
  Val    Loss=0.3183  F1=0.7341  Prec=0.7693  Recall=0.7074  AUROC=0.9259
  Val per-class AUROC: NORM=0.945  MI=0.929  STTC=0.927  CD=0.930  HYP=0.898
  Patience 10/12



Epoch 28/50  (lr=2.50e-04)
  Train  Loss=0.1838  F1=0.8259
  Val    Loss=0.3226  F1=0.7377  Prec=0.7637  Recall=0.7200  AUROC=0.9249
  Val per-class AUROC: NORM=0.944  MI=0.932  STTC=0.923  CD=0.928  HYP=0.898
  Patience 11/12



Epoch 29/50  (lr=2.33e-04)
  Train  Loss=0.1823  F1=0.8280
  Val    Loss=0.3357  F1=0.7305  Prec=0.7755  Recall=0.6990  AUROC=0.9266
  Val per-class AUROC: NORM=0.945  MI=0.927  STTC=0.925  CD=0.934  HYP=0.902
  Patience 12/12

Early stopping at epoch 29

Loading best model for evaluation...

Tuning per-class thresholds on validation set...
    NORM : 0.400
    MI   : 0.610
    STTC : 0.300
    CD   : 0.490
    HYP  : 0.310

Test with threshold 0.5:


  Macro F1=0.7484  Prec=0.7732  Recall=0.7325

Test with tuned per-class thresholds:



FINAL TEST RESULTS — CNN + Transformer (Proposed model 1)
  Loss      : 0.2876
  Macro F1  : 0.7529
  Macro AUROC: 0.9276
  Macro Prec: 0.7383
  Macro Rec : 0.7726
  Per-class AUROC:
    NORM : 0.9471
    MI   : 0.9246
    STTC : 0.9365
    CD   : 0.9249
    HYP  : 0.9050

Artifacts saved to ../checkpoint_proposed/
